# 🧠 Demographic Proxy Feature Engineering
### From E-Commerce / Financial Transaction Data

**Features engineered:** Age Group · Income Level · Education Background · Home Location · Work Location

> All features are **probabilistic proxies** inferred from behavioral signals — not ground truth labels.
> MCC lookup tables are derived from `mcc_gender_predictions.csv`.

---


## 📦 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

print("✅ Libraries loaded")


## 📂 1. Load Data

In [ ]:
# ── Load transaction data
df = pd.read_csv('your_transactions.csv')   # ← update path

# ── Load MCC lookup table
mcc = pd.read_csv('mcc_gender_predictions.csv')

# ── Data types
df['transaction_date']     = pd.to_datetime(df['transaction_date'])
df['merchant_category_id'] = df['merchant_category_id'].astype(int)
mcc['mcc_code']            = mcc['mcc_code'].astype(int)

# ── Filter completed transactions only
if 'transaction_status' in df.columns:
    before = len(df)
    df = df[df['transaction_status'].str.lower() == 'completed'].copy()
    print(f"   Filtered {before - len(df):,} non-completed transactions")

print(f"✅ Transactions loaded : {len(df):,} rows")
print(f"   Unique users        : {df['user_id'].nunique():,}")
print(f"   Date range          : {df['transaction_date'].min().date()} → {df['transaction_date'].max().date()}")
print(f"\nTransaction columns   : {df.columns.tolist()}")


## 🔍 2. Quick Data Sanity Check

In [ ]:
print("=== SHAPE ===")
print(f"Transactions: {df.shape}")
print(f"MCC table   : {mcc.shape}")

print("\n=== NULL COUNTS ===")
display(df.isnull().sum().to_frame('nulls').T)

print("\n=== KEY COLUMN SAMPLES ===")
for col in ['geo_location', 'user_agent', 'payment_method', 'merchant_name']:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts().head(5))


## ⏱️ 3. Time Feature Extraction
> Shared features used by all demographic proxies

In [ ]:
df['hour']         = df['transaction_date'].dt.hour
df['day_of_week']  = df['transaction_date'].dt.dayofweek   # 0=Mon, 6=Sun
df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
df['is_weekday']   = (~df['day_of_week'].isin([5, 6])).astype(int)
df['is_late_night']= (df['hour'].between(22, 23) | df['hour'].between(0, 2)).astype(int)
df['is_daytime']   = df['hour'].between(9, 17).astype(int)

# ── Quick visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Transaction Volume by Hour of Day')
axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Count')

df['day_of_week'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='darkorange',
    tick_label=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
axes[1].set_title('Transaction Volume by Day of Week')
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()
print("✅ Time features extracted")


## 🏷️ 4. MCC Lookup Tables
> Built from actual codes in `mcc_gender_predictions.csv`

In [ ]:
# ── AGE: YOUNG (18–30)
# Fast food, gaming, digital goods, music, sports, schools, candy, streaming
MCC_AGE_YOUNG = [
    # Fast Food
    5814, 1083, 5414,
    # Gaming & Video Games
    5818, 5823, 5824, 5826, 5854, 3639, 3920, 3921, 3923, 3943, 3947, 6284,
    # Digital Goods (streaming, apps, media)
    5817, 3975, 6460, 2748, 2743, 2744, 5849, 5844, 5864,
    # Music
    5733, 573,
    # Sports & Recreation
    5356, 5941, 5553, 5655, 5650, 714,
    # Schools / Students
    821, 822, 823, 824, 825, 828,
    # Candy & Confectionery
    5471, 5481, 5482, 5483, 5484, 5485, 5486, 5487, 5488, 5489,
    5440, 5441, 5442, 5443, 5444, 5445, 5446, 5447, 5448, 5449,
    5405, 5406, 5407, 5408, 5415, 5416, 5418, 5456, 5457, 5473,
    5476, 5477, 5478, 544,
    # Video Rental & Toy/Hobby
    3864, 3694, 5945, 5268,
]

# ── AGE: MATURE (45+)
# Insurance, medical, hospitals, mortgage, pension, funeral, real estate
MCC_AGE_MATURE = [
    # Hospitals
    808, 809, 3690, 3691, 3692, 3693, 3695, 3696, 3697, 3698,
    # Medical Services (3800–3899 range)
    *range(3800, 3900),
    883, 805,
    # Medical Labs
    807, 6322, 6323,
    # Insurance (all 63xx range)
    *[c for c in range(6300, 6420) if c in mcc['mcc_code'].values],
    # Mortgage & Real Estate
    6116, 6124, 6125, 6126, 6127, 6161, 6162, 6164, 6165,
    6166, 6167, 6168, 6169, 6176, 6112,
    # Funeral & Real Estate
    726, 651,
]

# ── INCOME: HIGH
# Jewelry, boats, aircraft, antiques, country clubs, fur, art, cigars, hotels
MCC_INCOME_HIGH = [
    # Jewelry & Precious Stones
    5944, 5972, 5096, 5094, 5095, 5091, 5097, 5976, 5952,
    1434, 1437, 1470, 1473, 1474, 1475, 1476, 1478, 1486, 1487,
    1680, 1687, 1867, 2144, 2146, 2147, 2178, 2567, 2637, 2643,
    2653, 2657, 2663, 2667, 2669, 2681, 2683, 2689,
    # Boats & Marine
    5563, 5551, 5557, 5558, 5565, 5591, 5518, 4422, 4425, 4457, 4473,
    # Aircraft & Aviation
    4505, 4527, 4535, 4537, 4544, 4546, 4547, 4564, 4566,
    4592, 4597, 4740, 1161,
    # Antique Shops
    5755, 5752, 5937, 5932, 5930, 5934, 5936, 5938, 5939,
    5560, 5561, 5703, 5704, 5708, 5709, 5743, 5759,
    5760, 5761, 5762, 5763, 5769, 5831, 5836, 5837, 5839, 5973,
    5324, 5327, 5342, 5347, 5360, 5394, 5396, 765, 575,
    # Art Galleries
    5092, 5093, 5098, 5971,
    # Country Clubs & Golf
    3906, 1483, 1226,
    # Fur Stores
    5683, 5636, 5680, 5681, 5682, 5684, 5685, 5686, 5687, 5688, 5689, 2381,
    # Cigars
    5873, 5993, 5808,
    # Hotels & Resorts
    3499, 3500, 3599, 3799, 3299,
    # Spas & Ski Resorts
    6302, 6382, 6387, 2596,
]

# ── INCOME: BUDGET
# Variety stores, wholesale clubs, pawn shops, used merchandise, discount
MCC_INCOME_BUDGET = [
    # Variety Stores
    5331, 5322, 5303, 5330, 5332, 5333, 5334, 5335, 5336, 5338, 5339, 533, 5803,
    # Wholesale Clubs
    5300, 5301, 5302, 5306, 5470, 5957, 5815, 1464, 530,
    # Pawn Shops
    5933, 5881, 5893, 5872, 5870, 5757,
    # Used Merchandise
    5794, 5931, 5774, 5776, 5790, 5793, 5795, 5626, 5702,
    # Discount Stores
    5310, 5317, 5316, 5328,
    # Used Cars
    551, 5555, 5589,
    # Convenience Stores
    5499, 5491, 5490, 5479, 5419, 1191, 1099, 1118, 1119, 1390, 1809, 5819,
    # Fast Food
    5814, 1083, 5414,
]

# ── EDUCATION: HIGH
# Bookstores, schools, universities, computer software, stationery, publishing
MCC_EDUCATION_HIGH = [
    # Schools & Colleges
    821, 822, 823, 824, 825, 828,
    # Bookstores
    2703, 2706, 2708, 2709, 2730, 2731, 2732, 2733, 2734,
    2735, 2736, 2737, 2738, 2739, 2755, 2756, 2770, 2771,
    2772, 2775, 2778, 2780, 2783, 2786, 2788, 5942,
    # Books, Periodicals, Newspapers
    5192, 5194, 5038, 2784, 2789, 2781,
    # Digital Goods — Books/Education
    6460, 2748, 2743, 2744, 5849, 5844, 5864,
    # Computer Software
    5734, 5751, 5758, 5738, 5746, 5754, 5788, 5180, 5181, 5182, 5183,
    # Stationery & Office Supply
    5943, 5996, 5946, 5117, 5111, 5112, 5113, 5116, 5806, 511,
    # Publishing & Printing
    2750, 2751, 2752, 2753, 2754, 2715, 2716, 2717, 2718, 2719,
    # Computer / Info Services
    4816, 5780, 737,
    # Tutoring & Professional Services
    829, 872,
]

print("✅ MCC lookup tables defined")
print(f"   Young MCC codes     : {len(set(MCC_AGE_YOUNG)):,}")
print(f"   Mature MCC codes    : {len(set(MCC_AGE_MATURE)):,}")
print(f"   High income codes   : {len(set(MCC_INCOME_HIGH)):,}")
print(f"   Budget codes        : {len(set(MCC_INCOME_BUDGET)):,}")
print(f"   Education codes     : {len(set(MCC_EDUCATION_HIGH)):,}")


## 🔗 5. Join MCC Signals to Transactions

In [ ]:
# Join MCC name
df = df.merge(
    mcc[['mcc_code', 'MCC_Code_name']].drop_duplicates('mcc_code'),
    left_on='merchant_category_id',
    right_on='mcc_code',
    how='left'
)

# Tag signals per transaction
df['is_young_mcc']         = df['merchant_category_id'].isin(MCC_AGE_YOUNG).astype(int)
df['is_mature_mcc']        = df['merchant_category_id'].isin(MCC_AGE_MATURE).astype(int)
df['is_high_income_mcc']   = df['merchant_category_id'].isin(MCC_INCOME_HIGH).astype(int)
df['is_budget_income_mcc'] = df['merchant_category_id'].isin(MCC_INCOME_BUDGET).astype(int)
df['is_edu_mcc']           = df['merchant_category_id'].isin(MCC_EDUCATION_HIGH).astype(int)

print("✅ MCC signals tagged")
signal_summary = pd.DataFrame({
    'Signal'     : ['Young MCC', 'Mature MCC', 'High Income MCC', 'Budget MCC', 'Education MCC'],
    'Transactions': [
        df['is_young_mcc'].sum(),
        df['is_mature_mcc'].sum(),
        df['is_high_income_mcc'].sum(),
        df['is_budget_income_mcc'].sum(),
        df['is_edu_mcc'].sum(),
    ],
    'Pct of Total': [
        f"{df['is_young_mcc'].mean()*100:.1f}%",
        f"{df['is_mature_mcc'].mean()*100:.1f}%",
        f"{df['is_high_income_mcc'].mean()*100:.1f}%",
        f"{df['is_budget_income_mcc'].mean()*100:.1f}%",
        f"{df['is_edu_mcc'].mean()*100:.1f}%",
    ]
})
display(signal_summary)


## 👶 6. Age Proxy
**Signals used:** MCC category + time of day + device type  
**Output:** `age_proxy_score` (0=young → 1=mature), `age_group` label

In [ ]:
# ── Device type from user_agent
def extract_device(ua):
    if pd.isna(ua): return 'unknown'
    ua = str(ua).lower()
    if any(x in ua for x in ['mobile', 'android', 'iphone', 'samsung']): return 'mobile'
    elif any(x in ua for x in ['tablet', 'ipad']): return 'tablet'
    else: return 'desktop'

df['device_type'] = df['user_agent'].apply(extract_device)

# ── Aggregate to user level
age_features = df.groupby('user_id').agg(
    pct_young_mcc   = ('is_young_mcc',    'mean'),
    pct_mature_mcc  = ('is_mature_mcc',   'mean'),
    pct_late_night  = ('is_late_night',   'mean'),
    pct_daytime     = ('is_daytime',      'mean'),
    pct_mobile      = ('device_type',     lambda x: (x == 'mobile').mean()),
    pct_desktop     = ('device_type',     lambda x: (x == 'desktop').mean()),
    total_txns      = ('transaction_amount', 'count'),
).reset_index()

# ── Weighted composite score (0=young, 1=mature)
age_features['age_proxy_score'] = (
    age_features['pct_mature_mcc']              * 0.35 +
    age_features['pct_daytime']                 * 0.25 +
    age_features['pct_desktop']                 * 0.20 +
    (1 - age_features['pct_young_mcc'])         * 0.10 +
    (1 - age_features['pct_late_night'])        * 0.10
)

# ── Age group label
def assign_age_group(score):
    if score >= 0.65:   return '45+'
    elif score >= 0.50: return '35-44'
    elif score >= 0.35: return '25-34'
    else:               return '18-24'

age_features['age_group'] = age_features['age_proxy_score'].apply(assign_age_group)

# ── Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
age_features['age_group'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Age Group Distribution'); axes[0].set_xlabel('Age Group')

axes[1].hist(age_features['age_proxy_score'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Age Proxy Score Distribution')
axes[1].set_xlabel('Score (0=young, 1=mature)')
plt.tight_layout(); plt.show()

print("✅ Age proxy computed")
display(age_features[['user_id','age_proxy_score','age_group']].head(8))


## 💰 7. Income Level
**Signals used:** Spend amount + MCC tier + payment method + promo sensitivity + loyalty  
**Output:** `income_score`, `income_tier` (Low / Mid / High)

In [ ]:
# ── Payment method scoring
def payment_score(method):
    if pd.isna(method): return 0.5
    method = str(method).lower()
    if any(x in method for x in ['credit', 'premium', 'gold', 'platinum']): return 1.0
    elif any(x in method for x in ['debit', 'prepaid']): return 0.4
    elif any(x in method for x in ['cod', 'cash']): return 0.2
    elif any(x in method for x in ['ewallet', 'gopay', 'ovo', 'dana', 'digital']): return 0.5
    return 0.5

df['payment_income_score'] = df['payment_method'].apply(payment_score)

# ── Aggregate to user level
income_features = df.groupby('user_id').agg(
    avg_transaction      = ('transaction_amount',    'mean'),
    median_transaction   = ('transaction_amount',    'median'),
    total_spend          = ('transaction_amount',    'sum'),
    std_transaction      = ('transaction_amount',    lambda x: x.std() if len(x) > 1 else 0),
    pct_high_income_mcc  = ('is_high_income_mcc',   'mean'),
    pct_budget_mcc       = ('is_budget_income_mcc', 'mean'),
    avg_payment_score    = ('payment_income_score',  'mean'),
    pct_discounted       = ('discount_applied',      'mean'),
    avg_promo_amount     = ('promo_amount',          'mean'),
    uses_loyalty         = ('loyalty_program',       lambda x: x.notna().mean()),
).reset_index()

# ── Spend percentile
income_features['spend_percentile'] = income_features['avg_transaction'].rank(pct=True)

# ── Composite income score (0=low, 1=high)
income_features['income_score'] = (
    income_features['spend_percentile']                        * 0.35 +
    income_features['avg_payment_score']                       * 0.25 +
    income_features['pct_high_income_mcc']                     * 0.15 +
    (1 - income_features['pct_budget_mcc'])                    * 0.10 +
    (1 - income_features['pct_discounted'].fillna(0))          * 0.10 +
    income_features['uses_loyalty']                            * 0.05
)

def assign_income_tier(score):
    if score >= 0.67:   return 'High'
    elif score >= 0.33: return 'Mid'
    else:               return 'Low'

income_features['income_tier'] = income_features['income_score'].apply(assign_income_tier)

# ── Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
income_features['income_tier'].value_counts().plot(
    kind='bar', ax=axes[0], color='darkorange', edgecolor='white')
axes[0].set_title('Income Tier Distribution'); axes[0].set_xlabel('Tier')

axes[1].hist(income_features['income_score'], bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Income Score Distribution')
axes[1].set_xlabel('Score (0=low, 1=high)')
plt.tight_layout(); plt.show()

print("✅ Income level computed")
display(income_features[['user_id','income_score','income_tier','avg_transaction','income_tier']].head(8))


## 🎓 8. Education Background
**Signals used:** MCC codes + merchant name keywords + spending sophistication  
**Output:** `education_score`, `education_level` (Basic / Moderate / Advanced)  
> ⚠️ **Low confidence** — treat as supporting feature only

In [ ]:
# ── Merchant name keyword signal
EDU_MERCHANT_KEYWORDS = [
    'book', 'bookstore', 'library', 'university', 'college', 'school',
    'academy', 'course', 'training', 'seminar', 'workshop', 'udemy',
    'coursera', 'skillshare', 'office', 'stationery', 'lab', 'research'
]

def merchant_edu_signal(name):
    if pd.isna(name): return 0
    name = str(name).lower()
    return int(any(kw in name for kw in EDU_MERCHANT_KEYWORDS))

df['is_edu_merchant'] = df['merchant_name'].apply(merchant_edu_signal)

# ── Aggregate to user level
edu_features = df.groupby('user_id').agg(
    pct_edu_mcc         = ('is_edu_mcc',            'mean'),
    pct_edu_merchant    = ('is_edu_merchant',        'mean'),
    category_diversity  = ('merchant_category_id',  'nunique'),
    pct_no_discount     = ('discount_applied',       lambda x: (x == 0).mean()),
    avg_merchant_rating = ('merchant_rating',        'mean'),
).reset_index()

# ── Normalize category diversity
edu_features['diversity_pct'] = edu_features['category_diversity'].rank(pct=True)

# ── Education score (0=low, 1=high)
edu_features['education_score'] = (
    edu_features['pct_edu_mcc']                     * 0.40 +
    edu_features['pct_edu_merchant']                * 0.30 +
    edu_features['diversity_pct']                   * 0.15 +
    edu_features['pct_no_discount'].fillna(0)       * 0.15
)

def assign_education_level(score):
    if score >= 0.60:   return 'Advanced'
    elif score >= 0.30: return 'Moderate'
    else:               return 'Basic'

edu_features['education_level'] = edu_features['education_score'].apply(assign_education_level)

# ── Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
edu_features['education_level'].value_counts().plot(
    kind='bar', ax=axes[0], color='seagreen', edgecolor='white')
axes[0].set_title('Education Level Distribution'); axes[0].set_xlabel('Level')

axes[1].hist(edu_features['education_score'], bins=30, color='seagreen', edgecolor='white')
axes[1].set_title('Education Score Distribution')
axes[1].set_xlabel('Score (0=basic, 1=advanced)')
plt.tight_layout(); plt.show()

print("✅ Education proxy computed  ⚠️  Low confidence")
display(edu_features[['user_id','education_score','education_level']].head(8))


## 🏠 9. Home Location
**Logic:** Most frequent `geo_location` during **weekends** or **evenings (18:00–23:00)**  
**Output:** `home_location`, `home_location_confidence`, `home_location_reliable`

In [ ]:
def most_frequent_geo(series):
    counts = series.value_counts()
    return counts.index[0] if len(counts) > 0 else None

def geo_confidence(series):
    counts = series.value_counts()
    return counts.iloc[0] / len(series) if len(counts) > 0 else 0.0

# ── Filter to home-time transactions
df['is_home_time'] = (
    (df['is_weekend'] == 1) | df['hour'].between(18, 23)
).astype(int)

home_df = df[df['is_home_time'] == 1]

home_location = home_df.groupby('user_id')['geo_location'].agg([
    ('home_location',             most_frequent_geo),
    ('home_location_confidence',  geo_confidence),
    ('home_txn_count',            'count'),
]).reset_index()

# ── Reliability flag: confidence ≥ 0.4 AND at least 3 transactions
home_location['home_location_reliable'] = (
    (home_location['home_location_confidence'] >= 0.4) &
    (home_location['home_txn_count'] >= 3)
).astype(int)

# ── Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
home_location['home_location_confidence'].hist(
    bins=30, ax=axes[0], color='mediumpurple', edgecolor='white')
axes[0].set_title('Home Location Confidence')
axes[0].set_xlabel('Confidence Score'); axes[0].axvline(0.4, color='red', linestyle='--', label='Threshold')
axes[0].legend()

home_location['home_location_reliable'].value_counts().plot(
    kind='bar', ax=axes[1], color=['salmon','mediumpurple'],
    edgecolor='white', tick_label=['Unreliable','Reliable'])
axes[1].set_title('Home Location Reliability')
plt.tight_layout(); plt.show()

print("✅ Home location computed")
print(f"   Reliable home locations : {home_location['home_location_reliable'].sum():,} / {len(home_location):,} users")
print(f"   Avg confidence          : {home_location['home_location_confidence'].mean():.2f}")
display(home_location.head(8))


## 🏢 10. Work Location
**Logic:** Most frequent `geo_location` during **weekdays, 09:00–17:00**  
**Output:** `work_location`, `is_remote_worker` flag

In [ ]:
# ── Filter to work-time transactions
df['is_work_time'] = (
    (df['is_weekday'] == 1) & df['hour'].between(9, 17)
).astype(int)

work_df = df[df['is_work_time'] == 1]

work_location = work_df.groupby('user_id')['geo_location'].agg([
    ('work_location',             most_frequent_geo),
    ('work_location_confidence',  geo_confidence),
    ('work_txn_count',            'count'),
]).reset_index()

# ── Cross-check: if work geo == home geo → likely remote worker
work_location = work_location.merge(
    home_location[['user_id', 'home_location']], on='user_id', how='left'
)
work_location['is_remote_worker'] = (
    work_location['work_location'] == work_location['home_location']
).astype(int)

work_location['work_location_reliable'] = (
    (work_location['work_location_confidence'] >= 0.4) &
    (work_location['work_txn_count'] >= 3)
).astype(int)

# ── Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
work_location['work_location_confidence'].hist(
    bins=30, ax=axes[0], color='tomato', edgecolor='white')
axes[0].set_title('Work Location Confidence')
axes[0].set_xlabel('Confidence Score'); axes[0].axvline(0.4, color='navy', linestyle='--', label='Threshold')
axes[0].legend()

work_location['is_remote_worker'].value_counts().plot(
    kind='bar', ax=axes[1], color=['steelblue','tomato'],
    edgecolor='white', tick_label=['Office','Remote'])
axes[1].set_title('Remote vs Office Workers')
plt.tight_layout(); plt.show()

print("✅ Work location computed")
print(f"   Likely remote workers   : {work_location['is_remote_worker'].sum():,} users")
print(f"   Reliable work locations : {work_location['work_location_reliable'].sum():,} users")
display(work_location.head(8))


## 🔗 11. Merge All Features → User Profile

In [ ]:
user_profile = (
    age_features[[
        'user_id', 'age_proxy_score', 'age_group',
        'pct_young_mcc', 'pct_mature_mcc', 'pct_mobile'
    ]]
    .merge(income_features[[
        'user_id', 'income_score', 'income_tier',
        'avg_transaction', 'total_spend',
        'pct_high_income_mcc', 'pct_budget_mcc'
    ]], on='user_id', how='left')
    .merge(edu_features[[
        'user_id', 'education_score', 'education_level',
        'pct_edu_mcc', 'category_diversity'
    ]], on='user_id', how='left')
    .merge(home_location[[
        'user_id', 'home_location',
        'home_location_confidence', 'home_location_reliable'
    ]], on='user_id', how='left')
    .merge(work_location[[
        'user_id', 'work_location', 'work_location_confidence',
        'is_remote_worker', 'work_location_reliable'
    ]], on='user_id', how='left')
)

print(f"✅ User Profile shape: {user_profile.shape}")
display(user_profile.head(10))


## 📊 12. Demographic Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('User Demographic Proxy Summary', fontsize=16, fontweight='bold')

# Age
user_profile['age_group'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0,0], color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Group'); axes[0,0].set_xlabel('')

# Income
user_profile['income_tier'].value_counts().plot(
    kind='bar', ax=axes[0,1], color='darkorange', edgecolor='white')
axes[0,1].set_title('Income Tier'); axes[0,1].set_xlabel('')

# Education
user_profile['education_level'].value_counts().plot(
    kind='bar', ax=axes[0,2], color='seagreen', edgecolor='white')
axes[0,2].set_title('Education Level  ⚠️ Low confidence'); axes[0,2].set_xlabel('')

# Income score distribution
axes[1,0].hist(user_profile['income_score'], bins=30, color='darkorange', edgecolor='white')
axes[1,0].set_title('Income Score Distribution')

# Age score distribution
axes[1,1].hist(user_profile['age_proxy_score'], bins=30, color='steelblue', edgecolor='white')
axes[1,1].set_title('Age Score Distribution')

# Remote workers
user_profile['is_remote_worker'].value_counts().plot(
    kind='bar', ax=axes[1,2], color=['teal','tomato'],
    edgecolor='white', tick_label=['Office','Remote'])
axes[1,2].set_title('Remote vs Office Workers')

plt.tight_layout()
plt.show()

print("\n=== SUMMARY ===")
print(f"Total users profiled : {len(user_profile):,}")
print("\nAge Groups:\n",        user_profile['age_group'].value_counts().to_string())
print("\nIncome Tiers:\n",      user_profile['income_tier'].value_counts().to_string())
print("\nEducation Levels:\n",  user_profile['education_level'].value_counts().to_string())


## 💾 13. Save Output

In [ ]:
user_profile.to_csv('user_demographic_profile.csv', index=False)
print("✅ Saved → user_demographic_profile.csv")
print(f"   Rows    : {len(user_profile):,}")
print(f"   Columns : {user_profile.columns.tolist()}")


## 📋 14. Feature Confidence Reference

| Feature | Confidence | Primary Signal | Notes |
|---|---|---|---|
| **Income Tier** | 🟢 Medium-High | Spend amount + MCC tier | Most reliable proxy |
| **Home Location** | 🟢 Medium-High | Geo + evening/weekend filter | Depends on geo data quality |
| **Work Location** | 🟡 Medium | Geo + weekday daytime filter | Remote workers skew results |
| **Age Group** | 🟡 Medium | MCC category + device type | Time signals help |
| **Education Level** | 🔴 Low | MCC bookstores/schools | Very weak signal — use cautiously |

> ⚠️ All features are **probabilistic proxies**. Validate against any available ground truth before using in production models.
